<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/vector_stores/TopKDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TopK Vector Store

[TopK](https://topk.io) is a search database for AI applications. It has built-in **semantic search**, with embeddings computed and reranked on the server, plus **BM25 keyword search**, **dense**, **sparse** and **multi-vector** search, **metadata filtering** and **custom scoring**. All of them can be combined in **one query**.

This notebook shows every TopK feature that `TopKVectorStore` exposes in LlamaIndex, each with a runnable example and a link to the matching [TopK guide](https://docs.topk.io):

1. [Semantic search with no embedding model](#semantic-search-with-no-embedding-model)
2. [Keyword search (BM25)](#keyword-search-bm25)
3. [True hybrid search](#true-hybrid-search)
4. [Custom scoring expressions](#custom-scoring-expressions)
5. [Metadata filters](#metadata-filters)
6. [Dense vectors from your own embedding model](#dense-vectors-from-your-own-embedding-model)
7. [Sparse vectors (SPLADE)](#sparse-vectors-splade)
8. [Multi-vector search (ColBERT, MaxSim)](#multi-vector-search-colbert-maxsim)
9. [Partitions for multi-tenancy](#partitions-for-multi-tenancy)
10. [Consistency](#consistency)
11. [Async](#async)
12. [The full TopK query language](#the-full-topk-query-language)

## Setup

If you're opening this notebook on Colab, you will need to install the packages. The FastEmbed packages are only needed for the dense, sparse and multi-vector sections; they run small models locally (`llama-index-sparse-embeddings-fastembed` currently needs Python 3.12 or lower).

In [ ]:
%pip install llama-index-vector-stores-topk llama-index-embeddings-fastembed llama-index-sparse-embeddings-fastembed fastembed

Create an API key in the [TopK console](https://console.topk.io) and pick your [region](https://docs.topk.io/regions).

In [1]:
import os
from getpass import getpass

if not os.environ.get("TOPK_API_KEY"):
    os.environ["TOPK_API_KEY"] = getpass("TopK API key: ")
os.environ.setdefault("TOPK_REGION", "aws-us-east-1-elastica")

'aws-us-east-1-elastica'

A small corpus with metadata. Collection names get a random suffix so the notebook can be re-run: TopK doesn't allow recreating a deleted collection under the same name right away.

In [2]:
import uuid

from llama_index.core import Document

RUN = uuid.uuid4().hex[:6]

documents = [
    Document(
        id_="rust",
        text="Rust guarantees memory safety without a garbage collector.",
        metadata={"lang": "rust", "year": 2015, "tags": ["systems", "safety"], "stars": 4.9},
    ),
    Document(
        id_="python",
        text="Python is known for readable syntax and scientific libraries.",
        metadata={"lang": "python", "year": 1991, "tags": ["scripting", "data"], "stars": 4.7},
    ),
    Document(
        id_="go",
        text="Go has a garbage collector and built-in goroutines for concurrency.",
        metadata={"lang": "go", "year": 2012, "tags": ["systems", "concurrency"], "stars": 4.5},
    ),
    Document(
        id_="haskell",
        text="Haskell is a purely functional language with lazy evaluation.",
        metadata={"lang": "haskell", "year": 1990, "tags": ["functional"], "stars": 4.1},
    ),
    Document(
        id_="sql",
        text="SQL queries relational databases with declarative statements.",
        metadata={"lang": "sql", "year": 1974, "tags": ["data"], "stars": 4.0},
    ),
]


def show(results):
    for r in results:
        print(f"{r.score:8.3f}  {r.node.metadata['lang']:8}  {r.node.get_content()}")

## Semantic search with no embedding model

TopK [embeds documents and queries on the server](https://docs.topk.io/guides/semantic-search) and reranks results, so you don't need an embedding model, an embedding API key or an embedding pipeline.

`VectorStoreIndex` always asks for an `embed_model`. Pass `TopKServerSideEmbedding()`, a placeholder that makes no network calls, so that LlamaIndex doesn't fall back to its default OpenAI model. (Setting `Settings.embed_model = None` also works.)

A value in TopK's semantic index can be at most 32,768 characters. Splitting documents into passage-sized chunks, for example with `SentenceSplitter(chunk_size=512)`, still gives more precise retrieval.

In [3]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.topk import TopKServerSideEmbedding, TopKVectorStore

vector_store = TopKVectorStore(collection_name=f"llamaindex-demo-{RUN}")

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=StorageContext.from_defaults(vector_store=vector_store),
    embed_model=TopKServerSideEmbedding(),
    transformations=[SentenceSplitter(chunk_size=512)],
)

show(index.as_retriever(similarity_top_k=2).retrieve("memory safe systems programming"))

   2.694  rust      Rust guarantees memory safety without a garbage collector.
   2.385  go        Go has a garbage collector and built-in goroutines for concurrency.


The retriever and query engine work as usual. To answer questions with an LLM, pass one to `as_query_engine(llm=...)`.

To reconnect to an existing collection later, build the index from the store:

In [4]:
index = VectorStoreIndex.from_vector_store(
    TopKVectorStore(collection_name=f"llamaindex-demo-{RUN}"),
    embed_model=TopKServerSideEmbedding(),
)
show(index.as_retriever(similarity_top_k=1).retrieve("lazy functional language"))

   2.880  haskell   Haskell is a purely functional language with lazy evaluation.


## Keyword search (BM25)

`vector_store_query_mode="text_search"` runs TopK's [BM25 keyword search](https://docs.topk.io/guides/keyword-search) on the same text field. Only nodes that contain at least one query term are returned. Pass `keyword_match_all=True` to require every term.

In [5]:
show(index.as_retriever(vector_store_query_mode="text_search").retrieve("garbage collector"))

   1.680  go        Go has a garbage collector and built-in goroutines for concurrency.
   1.680  rust      Rust guarantees memory safety without a garbage collector.


In [6]:
retriever = index.as_retriever(
    vector_store_query_mode="text_search",
    vector_store_kwargs={"keyword_match_all": True},
)
show(retriever.retrieve("garbage goroutines"))

   2.170  go        Go has a garbage collector and built-in goroutines for concurrency.


## True hybrid search

`vector_store_query_mode="hybrid"` runs [true hybrid search](https://docs.topk.io/guides/true-hybrid-search): every signal the store is configured for is scored in **one TopK query** and combined with weights. There is no second round trip and no client-side fusion. Here the signals are semantic similarity and BM25; with dense, sparse or multi-vector models configured, they join the same query (see below).

BM25 boosts the nodes that contain query terms without excluding the others. Pass `keyword_required=True` to keep only nodes with at least one term.

In [7]:
show(index.as_retriever(vector_store_query_mode="hybrid", similarity_top_k=3).retrieve("garbage collector"))

   3.423  go        Go has a garbage collector and built-in goroutines for concurrency.
   3.416  rust      Rust guarantees memory safety without a garbage collector.
   1.345  sql       SQL queries relational databases with declarative statements.


`alpha` follows LlamaIndex's convention: `1.0` uses only the vector signals (semantic, dense, multi-vector) and `0.0` only the lexical ones (BM25, sparse). For finer control, pass per-signal `weights`. Semantic and BM25 scores have different ranges, so print a few real scores before choosing weights.

In [8]:
retriever = index.as_retriever(vector_store_query_mode="hybrid", alpha=0.8, similarity_top_k=3)
show(retriever.retrieve("garbage collector"))

   1.731  go        Go has a garbage collector and built-in goroutines for concurrency.
   1.725  rust      Rust guarantees memory safety without a garbage collector.
   1.076  sql       SQL queries relational databases with declarative statements.


In [9]:
retriever = index.as_retriever(
    vector_store_query_mode="hybrid",
    similarity_top_k=3,
    vector_store_kwargs={"weights": {"semantic": 1.0, "keyword": 0.2}, "keyword_required": True},
)
show(retriever.retrieve("garbage collector"))

   2.079  go        Go has a garbage collector and built-in goroutines for concurrency.
   2.072  rust      Rust guarantees memory safety without a garbage collector.


`vector_store_query_mode="semantic_hybrid"` is semantic search plus BM25 only, even when the store also has client-side vectors.

In [10]:
show(index.as_retriever(vector_store_query_mode="semantic_hybrid", similarity_top_k=2).retrieve("garbage collector"))

   3.423  go        Go has a garbage collector and built-in goroutines for concurrency.
   3.416  rust      Rust guarantees memory safety without a garbage collector.


## Custom scoring expressions

Pass `ranking` to sort results by your own [scoring expression](https://docs.topk.io/concepts#custom-scoring). It can combine the search scores with metadata fields, and it runs inside TopK. `ranking` is a TopK expression, or a callable that receives the score fields (`s.semantic`, `s.keyword`, `s.dense`, `s.sparse`, `s.multi_vector`) and returns one. Metadata fields are under `metadata.`.

A BM25 score is null for nodes without a query term, so wrap it in `.coalesce(0.0)`. The scores returned with each node remain the search scores, not the ranking value.

In [11]:
from topk_sdk.query import field

retriever = index.as_retriever(
    vector_store_query_mode="hybrid",
    similarity_top_k=3,
    vector_store_kwargs={
        # Relevance, plus a boost for highly starred languages.
        "ranking": lambda s: s.semantic
        + s.keyword.coalesce(0.0) * 0.2
        + field("metadata.stars") * 2.0,
    },
)
show(retriever.retrieve("programming language for systems"))

   2.506  python    Python is known for readable syntax and scientific libraries.
   2.466  go        Go has a garbage collector and built-in goroutines for concurrency.
   1.536  rust      Rust guarantees memory safety without a garbage collector.


## Metadata filters

LlamaIndex `MetadataFilters` are translated into TopK [filter expressions](https://docs.topk.io/collections/query#filtering) that run in the same query as the search. Every `FilterOperator` and every `FilterCondition` is supported:

| LlamaIndex | TopK |
|---|---|
| `==`, `!=`, `>`, `>=`, `<`, `<=` | `eq`, `ne`, `gt`, `gte`, `lt`, `lte` |
| `in`, `nin` | `in_`, `not_(in_)` |
| `contains` (list contains value) | `contains` |
| `any`, `all` (list contains any/all values) | `contains` joined with `or`/`and` |
| `text_match` | `contains` (case-sensitive substring) |
| `text_match_insensitive` | `regexp_match(..., "i")` |
| `is_empty` | `is_null()` |
| `and`, `or`, nested `MetadataFilters` | `and_`, `or_` |
| `not` (none of the filters match) | `not_(... or ...)` |

In [12]:
from llama_index.core.vector_stores import (
    FilterCondition,
    FilterOperator,
    MetadataFilter,
    MetadataFilters,
)

filters = MetadataFilters(
    filters=[
        MetadataFilter(key="tags", value=["systems", "data"], operator=FilterOperator.ANY),
        MetadataFilters(
            filters=[
                MetadataFilter(key="year", value=2000, operator=FilterOperator.GTE),
                MetadataFilter(key="lang", value="PYTH", operator=FilterOperator.TEXT_MATCH_INSENSITIVE),
            ],
            condition=FilterCondition.OR,
        ),
    ],
    condition=FilterCondition.AND,
)
show(index.as_retriever(filters=filters, similarity_top_k=5).retrieve("programming language"))

   1.975  python    Python is known for readable syntax and scientific libraries.
   1.687  go        Go has a garbage collector and built-in goroutines for concurrency.
   1.144  rust      Rust guarantees memory safety without a garbage collector.


In [13]:
not_go_or_rust = MetadataFilters(
    filters=[
        MetadataFilter(key="lang", value="go"),
        MetadataFilter(key="lang", value="rust"),
    ],
    condition=FilterCondition.NOT,
)
show(index.as_retriever(filters=not_go_or_rust, similarity_top_k=5).retrieve("garbage collector"))

   1.345  sql       SQL queries relational databases with declarative statements.
   1.277  haskell   Haskell is a purely functional language with lazy evaluation.
   1.115  python    Python is known for readable syntax and scientific libraries.


Filters also work for `get_nodes` and `delete_nodes`, and you can pass any TopK expression with `topk_filter`, for example a `match()` on the text:

In [14]:
from topk_sdk.query import match

retriever = index.as_retriever(
    similarity_top_k=5,
    vector_store_kwargs={"topk_filter": (field("metadata.year") < 2000) & (field("metadata.stars") > 4.05)},
)
show(retriever.retrieve("programming language"))
sql_nodes = vector_store.get_nodes(filters=MetadataFilters(filters=[MetadataFilter(key="lang", value="sql")]))
print([(n.ref_doc_id, n.get_content()) for n in sql_nodes])

   1.975  python    Python is known for readable syntax and scientific libraries.
   1.828  haskell   Haskell is a purely functional language with lazy evaluation.
[('sql', 'SQL queries relational databases with declarative statements.')]


## Dense vectors from your own embedding model

To use [dense vector search](https://docs.topk.io/guides/vector-search) with any LlamaIndex embedding model, create the store with `embed_dim`. TopK then stores the embeddings LlamaIndex computes in a vector index (`distance_metric` can be `"cosine"`, `"dot_product"` or `"euclidean"`), and `default` mode queries use them. The text field keeps its semantic index, so `hybrid` mode combines server-side semantic search, your dense vectors and BM25 in one query.

In [15]:
from llama_index.embeddings.fastembed import FastEmbedEmbedding

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")  # 384 dimensions

dense_store = TopKVectorStore(collection_name=f"llamaindex-demo-dense-{RUN}", embed_dim=384)
dense_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=StorageContext.from_defaults(vector_store=dense_store),
    embed_model=embed_model,
)

show(dense_index.as_retriever(similarity_top_k=2).retrieve("memory safe systems programming"))

   0.746  rust      Rust guarantees memory safety without a garbage collector.
   0.625  go        Go has a garbage collector and built-in goroutines for concurrency.


In [16]:
retriever = dense_index.as_retriever(
    vector_store_query_mode="hybrid",
    similarity_top_k=3,
    vector_store_kwargs={"weights": {"semantic": 1.0, "dense": 2.0, "keyword": 0.1}},
)
show(retriever.retrieve("garbage collector"))

   3.393  go        Go has a garbage collector and built-in goroutines for concurrency.
   3.315  rust      Rust guarantees memory safety without a garbage collector.
   2.326  sql       SQL queries relational databases with declarative statements.


## Sparse vectors (SPLADE)

TopK's [sparse vector search](https://docs.topk.io/guides/sparse-vector-search) is exact, with 100% recall, and works with learned sparse models such as SPLADE. Pass any LlamaIndex sparse embedding model as `sparse_embedding_model`. The store embeds node text when writing and the query when searching, and `vector_store_query_mode="sparse"` searches the sparse vectors.

In [17]:
from llama_index.sparse_embeddings.fastembed import FastEmbedSparseEmbedding

sparse_store = TopKVectorStore(
    collection_name=f"llamaindex-demo-sparse-{RUN}",
    sparse_embedding_model=FastEmbedSparseEmbedding(model_name="prithivida/Splade_PP_en_v1"),
)
sparse_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=StorageContext.from_defaults(vector_store=sparse_store),
    embed_model=TopKServerSideEmbedding(),
)

show(sparse_index.as_retriever(vector_store_query_mode="sparse", similarity_top_k=2).retrieve("memory safety"))

  18.185  rust      Rust guarantees memory safety without a garbage collector.
   0.410  go        Go has a garbage collector and built-in goroutines for concurrency.


## Multi-vector search (ColBERT, MaxSim)

TopK's [multi-vector search](https://docs.topk.io/guides/multi-vector-search) stores one vector per token and scores documents with MaxSim (late interaction, as in ColBERT). LlamaIndex has no interface for multi-vector models, so the package includes `BaseMultiVectorEmbedding` and a FastEmbed implementation, `FastEmbedMultiVectorEmbedding`.

There is no LlamaIndex query mode for multi-vector search, so select it with `search_mode`. In `hybrid` mode it joins the other signals: this store combines server-side semantic search, BM25, SPLADE and ColBERT in **one** query.

In [18]:
from llama_index.vector_stores.topk import FastEmbedMultiVectorEmbedding

all_signals_store = TopKVectorStore(
    collection_name=f"llamaindex-demo-multi-{RUN}",
    sparse_embedding_model=FastEmbedSparseEmbedding(model_name="prithivida/Splade_PP_en_v1"),
    multi_vector_embedding_model=FastEmbedMultiVectorEmbedding(),  # answerai-colbert-small-v1
)
multi_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=StorageContext.from_defaults(vector_store=all_signals_store),
    embed_model=TopKServerSideEmbedding(),
)

retriever = multi_index.as_retriever(similarity_top_k=2, vector_store_kwargs={"search_mode": "multi_vector"})
show(retriever.retrieve("relational database queries"))

  31.322  sql       SQL queries relational databases with declarative statements.
  30.571  go        Go has a garbage collector and built-in goroutines for concurrency.


In [19]:
retriever = multi_index.as_retriever(
    vector_store_query_mode="hybrid",
    similarity_top_k=3,
    vector_store_kwargs={"weights": {"semantic": 1.0, "keyword": 0.1, "sparse": 0.2, "multi_vector": 0.5}},
)
show(retriever.retrieve("lazy evaluation in functional programming"))

  23.287  haskell   Haskell is a purely functional language with lazy evaluation.
  17.586  sql       SQL queries relational databases with declarative statements.
  17.023  go        Go has a garbage collector and built-in goroutines for concurrency.


## Partitions for multi-tenancy

[Partitions](https://docs.topk.io/concepts#multi-tenancy) are isolated sub-collections that share a schema. A query against one partition never returns another partition's documents, which is safer than a `tenant_id` filter. Scope a store to a partition with `partition=`, or pass `partition` per call. A partition is created on its first write, and reading a partition that doesn't exist returns no results.

In [20]:
tenants = f"llamaindex-demo-tenants-{RUN}"
acme = TopKVectorStore(collection_name=tenants, partition="acme")
globex = TopKVectorStore(collection_name=tenants, partition="globex")

from llama_index.core.schema import TextNode

acme.add([TextNode(text="Acme's refund window is 30 days.")])
globex.add([TextNode(text="Globex offers refunds within 14 days.")])

acme_index = VectorStoreIndex.from_vector_store(acme, embed_model=TopKServerSideEmbedding())
for r in acme_index.as_retriever(similarity_top_k=5).retrieve("refund policy"):
    print(r.node.get_content())

print(acme.list_partitions())

Acme's refund window is 30 days.
['acme', 'globex']


One store can also serve several tenants by passing `partition` per call, for example `vector_store_kwargs={"partition": "globex"}` on a retriever. `delete_partition()` removes a tenant's data. Deletion takes about a minute, and the same partition name can't be written to again right away.

In [21]:
retriever = acme_index.as_retriever(vector_store_kwargs={"partition": "globex"})
print([r.node.get_content() for r in retriever.retrieve("refund policy")])

['Globex offers refunds within 14 days.']


## Consistency

TopK has three [read consistency](https://docs.topk.io/concepts#read-consistency) modes:

- `"balanced"` (default) reads indexed data plus recent writes, which appear in well under a second.
- `"indexed"` reads only compacted, indexed data, for the lowest latency and cost. Recent writes aren't visible until they are compacted.
- `"strong"` always includes the latest writes, at a higher latency.

Set a default with `TopKVectorStore(consistency=...)`, or pass `consistency` per query.

Every write returns an [LSN](https://docs.topk.io/collections/query#lsn-based-consistency). With `read_your_writes=True` (the default), the store passes the LSN of its latest write in a partition to later reads of that partition, so a query always sees what the same store just wrote. You can also pass an LSN explicitly, for example from another process.

In [22]:
fresh = TopKVectorStore(collection_name=f"llamaindex-demo-consistency-{RUN}")
fresh.add([TextNode(id_="zebra", text="Zebras have black and white stripes.")])
print("LSN of the last write:", fresh.last_lsn)

from llama_index.core.vector_stores import VectorStoreQuery

# Read-your-writes: the store passes its last LSN automatically.
print([n.node_id for n in fresh.query(VectorStoreQuery(query_str="striped animal")).nodes])

# Another store, without read-your-writes, waits for a given LSN.
reader = TopKVectorStore(collection_name=fresh.collection_name, read_your_writes=False)
print([n.node_id for n in reader.query(VectorStoreQuery(query_str="striped animal"), lsn=fresh.last_lsn).nodes])

# Strong and balanced consistency, per query.
for mode in ["strong", "balanced"]:
    print(mode, [n.node_id for n in reader.query(VectorStoreQuery(query_str="striped animal"), consistency=mode).nodes])

LSN of the last write: 1


['zebra']


['zebra']
strong ['zebra']


balanced ['zebra']


`"indexed"` skips writes that are not compacted yet, which can take a few minutes. On a collection written seconds ago, like the one above, it typically returns nothing yet; on a collection under steady traffic, it serves the indexed data at the lowest cost. Use it when your writes are asynchronous and don't need to be visible right away.

In [23]:
print("indexed", [n.node_id for n in reader.query(VectorStoreQuery(query_str="striped animal"), consistency="indexed").nodes])

indexed []


## Async

Every operation has an async version: `async_add`, `aquery`, `aget_nodes`, `adelete`, `adelete_nodes` and `aclear`, so `aretrieve` and async query engines use TopK's async client.

In [24]:
async_store = TopKVectorStore(collection_name=f"llamaindex-demo-async-{RUN}")
await async_store.async_add(
    [TextNode(id_=d.id_, text=d.text, metadata=d.metadata) for d in documents]
)
async_index = VectorStoreIndex.from_vector_store(async_store, embed_model=TopKServerSideEmbedding())
show(await async_index.as_retriever(similarity_top_k=2, vector_store_query_mode="hybrid").aretrieve("garbage collector"))

   3.423  go        Go has a garbage collector and built-in goroutines for concurrency.
   3.416  rust      Rust guarantees memory safety without a garbage collector.


## The full TopK query language

For anything else, such as aggregations or pagination, `query_raw` runs any [TopK query](https://docs.topk.io/collections/query) against the store's collection and partition. Node metadata is stored under `metadata`.

In [25]:
from topk_sdk.query import fn, select

rows = vector_store.query_raw(
    select("text", score=fn.semantic_similarity("text", "memory safety"))
    .filter(field("metadata.year") > 2000)
    .sort(field("score"), asc=False)
    .limit(2)
)
for row in rows:
    print(f"{row['score']:.3f}  {row['text']}")

2.013  Rust guarantees memory safety without a garbage collector.
1.340  Go has a garbage collector and built-in goroutines for concurrency.


## Clean up

In [26]:
for store in [vector_store, dense_store, sparse_store, all_signals_store, acme, fresh, async_store]:
    store.delete_collection()

## Learn more

- TopK: https://topk.io
- Console and API keys: https://console.topk.io
- Documentation: https://docs.topk.io
- Package: https://github.com/topk-io/llama-index-vector-stores-topk